# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashritha-boop/fly_machine/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

This is a **ranking** problem: the editor needs the most useful pages near the top of a review queue. The observed proxy label is `1` when April 2026 impressions are below 80% of March impressions. It measures next-month decline, not whether a refresh would cause recovery.

I compare two probability-ranked classifiers: **Logistic Regression** as a readable first model, and **HistGradientBoosting** as a nonlinear challenger that can capture interactions among the five Week-3 measurements. Both use only March-known information; hash IDs and April impressions are never features. A more complex model earns its place only if it improves held-out-client precision at the review-budget cutoffs.

In [2]:
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("duckdb") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])

import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if not HF_TOKEN:
    raise RuntimeError(
        "This analysis uses the gated FlyRank warehouse. Add your read-only HF_TOKEN "
        "to Colab Secrets (or the HF_TOKEN environment variable) after accepting the "
        "dataset terms. No synthetic-label fallback is used."
    )

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
safe_token = HF_TOKEN.replace("'", "''")
con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Keep only complete March-to-April outcomes; April is label-only, never a feature.
base_sql = f"""
WITH march AS (
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_march,
        SUM(gsc_clicks) AS gsc_clicks_march,
        AVG(gsc_avg_position) AS gsc_avg_position_march,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END)
            AS ga4_sessions_march,
        CASE WHEN BOOL_OR(ga4_data_available IS TRUE) THEN 1 ELSE 0 END
            AS ga4_available_march
    FROM {FACT}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY content_hash_id, client_hash_id
),
april AS (
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_april
    FROM {FACT}
    WHERE report_date BETWEEN '2026-04-01' AND '2026-04-30'
    GROUP BY content_hash_id, client_hash_id
)
SELECT
    m.content_hash_id,
    m.client_hash_id,
    m.gsc_impressions_march,
    m.gsc_clicks_march,
    m.gsc_avg_position_march,
    m.ga4_sessions_march,
    m.ga4_available_march,
    DATEDIFF('day', c.content_created_date, DATE '2026-03-31')
        AS content_age_days,
    a.gsc_impressions_april,
    CASE WHEN a.gsc_impressions_april < 0.80 * m.gsc_impressions_march
         THEN 1 ELSE 0 END AS is_declining_label
FROM march m
JOIN april a
  ON m.content_hash_id = a.content_hash_id
 AND m.client_hash_id = a.client_hash_id
JOIN {CONTENT} c
  ON m.content_hash_id = c.content_hash_id
 AND m.client_hash_id = c.client_hash_id
WHERE m.gsc_impressions_march >= 10
"""
df_data = con.sql(base_sql).df()
df_data["ga4_sessions_march"] = df_data["ga4_sessions_march"].where(
    df_data["ga4_available_march"].eq(1)
)
df_data = df_data.dropna(
    subset=[
        "gsc_impressions_march",
        "gsc_clicks_march",
        "gsc_avg_position_march",
        "content_age_days",
        "is_declining_label",
    ]
).reset_index(drop=True)

assert not df_data.duplicated(["client_hash_id", "content_hash_id"]).any()
assert df_data["is_declining_label"].isin([0, 1]).all()
assert df_data["client_hash_id"].nunique() >= 5

print(
    f"Paired March/April frame: {len(df_data):,} content items across "
    f"{df_data['client_hash_id'].nunique():,} clients; "
    f"April decline-label rate: {df_data['is_declining_label'].mean():.3f}."
)
print(
    "GA4 unavailable in March: "
    f"{df_data['ga4_available_march'].eq(0).mean():.1%} of rows; "
    "those session values remain missing for imputation."
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Paired March/April frame: 143,206 content items across 45 clients; April decline-label rate: 0.517.
GA4 unavailable in March: 53.8% of rows; those session values remain missing for imputation.


## 2. Split design

I use **five-fold GroupKFold by client**. A page from a held-out client cannot appear in training, which is more honest for a queue intended to transfer across client portfolios than a random row split. Each fold uses March 2026 as the feature window and April 2026 only for the outcome; records without both months are excluded rather than silently labeled non-decliners.

Week 4 reported its rule's precision on the full frame, not a held-out-client split. Those original numbers are useful context, but are not directly comparable to this test. I recompute the unchanged Week-4 score inside each held-out fold, so rule and models see the same test rows and use the same ranking metrics here. The five-fold standard deviation is descriptive client-to-client variation, not a confidence interval.

In [3]:
import sklearn
from sklearn.model_selection import GroupKFold

# The contract's five measurements plus explicit measurement-availability flags.
df_data["position_available_march"] = (
    df_data["gsc_avg_position_march"].gt(0).astype(int)
)
FEATURES = [
    "gsc_impressions_march",
    "gsc_clicks_march",
    "gsc_avg_position_march",
    "ga4_sessions_march",
    "content_age_days",
    "ga4_available_march",
    "position_available_march",
]
FORBIDDEN_FEATURES = {
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions_april",
    "is_declining_label",
}
assert not (set(FEATURES) & FORBIDDEN_FEATURES)

X = df_data[FEATURES].copy()
y = df_data["is_declining_label"].astype(int).copy()
groups = df_data["client_hash_id"].copy()
fold_splits = list(GroupKFold(n_splits=5).split(X, y, groups=groups))

fold_checks = []
for fold_number, (train_idx, test_idx) in enumerate(fold_splits, start=1):
    train_clients = set(groups.iloc[train_idx])
    test_clients = set(groups.iloc[test_idx])
    assert train_clients.isdisjoint(test_clients)
    assert len(test_idx) >= 100
    fold_checks.append(
        {
            "fold": fold_number,
            "train_clients": len(train_clients),
            "test_clients": len(test_clients),
            "test_rows": len(test_idx),
            "test_base_rate": y.iloc[test_idx].mean(),
        }
    )

print(f"scikit-learn version: {sklearn.__version__}")
print("Grouped split and feature leakage checks passed.")
print(pd.DataFrame(fold_checks).to_string(index=False, float_format="%.3f"))

scikit-learn version: 1.6.1
Grouped split and feature leakage checks passed.
 fold  train_clients  test_clients  test_rows  test_base_rate
    1             39             6      28639           0.432
    2             37             8      28643           0.682
    3             34            11      28645           0.425
    4             36             9      28638           0.546
    5             34            11      28641           0.499


## 3. Train + compare vs my baseline

I refit Logistic Regression and HistGradientBoosting in each training fold. The Week-4 rule remains exactly `high impressions × stale × position >= 4`, ranked by `impressions / (position + 1)`. The table compares its held-out-fold ranking with both models using precision@20, @50, and @100, plus average precision. It also shows the held-out base rate; all methods are scored on the same rows in each fold.

The main operational metric is precision@50 because it matches a 50-page review budget. Scores are used to order items only; they are not presented as calibrated probabilities.

In [4]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42


def precision_at_k(y_true, scores, k):
    y_array = np.asarray(y_true)
    score_array = np.asarray(scores)
    take = min(k, len(y_array))
    order = np.argsort(-score_array, kind="mergesort")[:take]
    return float(y_array[order].mean())


position = df_data["gsc_avg_position_march"].fillna(0)
stale = df_data["content_age_days"].ge(180)
high_volume = df_data["gsc_impressions_march"].ge(500)
suboptimal_position = position.ge(4.0)
baseline_scores = (
    stale.astype(int)
    * high_volume.astype(int)
    * suboptimal_position.astype(int)
    * df_data["gsc_impressions_march"]
    / (position + 1.0)
).to_numpy()

model_templates = {
    "Logistic Regression": make_pipeline(
        SimpleImputer(strategy="median", keep_empty_features=True),
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=SEED),
    ),
    "HistGradientBoosting": make_pipeline(
        SimpleImputer(strategy="median", keep_empty_features=True),
        HistGradientBoostingClassifier(
            max_iter=120,
            max_leaf_nodes=15,
            min_samples_leaf=30,
            l2_regularization=1.0,
            random_state=SEED,
        ),
    ),
}

fold_metrics = []
oof_scores = {
    "Week-4 rule": np.full(len(df_data), np.nan),
    **{name: np.full(len(df_data), np.nan) for name in model_templates},
}
fitted_by_fold = {name: [] for name in model_templates}

for fold_number, (train_idx, test_idx) in enumerate(fold_splits, start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    fold_base = float(y_test.mean())
    baseline_test_scores = baseline_scores[test_idx]
    oof_scores["Week-4 rule"][test_idx] = baseline_test_scores
    fold_metrics.append(
        {
            "fold": fold_number,
            "method": "Week-4 rule",
            "base_rate": fold_base,
            "precision@20": precision_at_k(y_test, baseline_test_scores, 20),
            "precision@50": precision_at_k(y_test, baseline_test_scores, 50),
            "precision@100": precision_at_k(y_test, baseline_test_scores, 100),
            "average_precision": average_precision_score(y_test, baseline_test_scores),
        }
    )

    for name, template in model_templates.items():
        fitted = clone(template)
        fitted.fit(X_train, y_train)
        scores = fitted.predict_proba(X_test)[:, 1]
        oof_scores[name][test_idx] = scores
        fitted_by_fold[name].append((fitted, X_test.copy(), y_test.to_numpy()))
        fold_metrics.append(
            {
                "fold": fold_number,
                "method": name,
                "base_rate": fold_base,
                "precision@20": precision_at_k(y_test, scores, 20),
                "precision@50": precision_at_k(y_test, scores, 50),
                "precision@100": precision_at_k(y_test, scores, 100),
                "average_precision": average_precision_score(y_test, scores),
            }
        )

assert all(np.isfinite(scores).all() for scores in oof_scores.values())
metrics_by_fold = pd.DataFrame(fold_metrics)
summary_rows = []
base_rates = metrics_by_fold.drop_duplicates("fold")["base_rate"]
summary_rows.append(
    {
        "method": "Random-selection base rate",
        "precision@20": f"{base_rates.mean():.3f} ± {base_rates.std(ddof=1):.3f}",
        "precision@50": f"{base_rates.mean():.3f} ± {base_rates.std(ddof=1):.3f}",
        "precision@100": f"{base_rates.mean():.3f} ± {base_rates.std(ddof=1):.3f}",
        "average precision": "n/a",
    }
)
for name in ["Week-4 rule", *model_templates]:
    fold_group = metrics_by_fold.loc[metrics_by_fold["method"].eq(name)]
    summary_rows.append(
        {
            "method": name,
            "precision@20": f"{fold_group['precision@20'].mean():.3f} ± {fold_group['precision@20'].std(ddof=1):.3f}",
            "precision@50": f"{fold_group['precision@50'].mean():.3f} ± {fold_group['precision@50'].std(ddof=1):.3f}",
            "precision@100": f"{fold_group['precision@100'].mean():.3f} ± {fold_group['precision@100'].std(ddof=1):.3f}",
            "average precision": f"{fold_group['average_precision'].mean():.3f} ± {fold_group['average_precision'].std(ddof=1):.3f}",
        }
    )

comparison_table = pd.DataFrame(summary_rows)
print("Held-out-client results: mean ± SD across five identical folds")
print(comparison_table.to_string(index=False))
model_p50 = metrics_by_fold.loc[
    metrics_by_fold["method"].isin(model_templates)
].groupby("method")["precision@50"].mean()
best_model_name = model_p50.idxmax()
rule_p50 = metrics_by_fold.loc[
    metrics_by_fold["method"].eq("Week-4 rule"), "precision@50"
].mean()
print(
    f"\nHighest observed model mean at P@50: {best_model_name} "
    f"({model_p50[best_model_name]:.3f}); Week-4 rule: {rule_p50:.3f}."
)
print("SD describes variation across held-out client folds; it is not a confidence interval.")

Held-out-client results: mean ± SD across five identical folds
                    method  precision@20  precision@50 precision@100 average precision
Random-selection base rate 0.517 ± 0.105 0.517 ± 0.105 0.517 ± 0.105               n/a
               Week-4 rule 0.340 ± 0.246 0.392 ± 0.214 0.420 ± 0.211     0.512 ± 0.116
       Logistic Regression 0.700 ± 0.122 0.688 ± 0.157 0.700 ± 0.177     0.560 ± 0.159
      HistGradientBoosting 0.460 ± 0.277 0.480 ± 0.193 0.470 ± 0.209     0.561 ± 0.129

Highest observed model mean at P@50: Logistic Regression (0.688); Week-4 rule: 0.392.
SD describes variation across held-out client folds; it is not a confidence interval.


## 4. Errors and interpretation

I inspect out-of-fold mistakes from the model with the highest observed mean precision@50. The 0.5 score cutoff is only a simple way to label false positives and false negatives; scores are not calibrated. Three anonymous examples show the March measurements alongside the April outcome. I also check whether error rates differ by content age and use held-out-fold permutation importance to see which measurements support the ranking. These interpretations are directional, not causal.

In [5]:
from sklearn.inspection import permutation_importance


def precision_at_50_scorer(estimator, feature_frame, labels):
    scores = estimator.predict_proba(feature_frame)[:, 1]
    label_array = np.asarray(labels)
    take = min(50, len(label_array))
    selected = np.argpartition(scores, -take)[-take:]
    return float(label_array[selected].mean())


importance_by_fold = []
for fitted, X_test, y_test in fitted_by_fold[best_model_name]:
    result = permutation_importance(
        fitted,
        X_test,
        y_test,
        scoring=precision_at_50_scorer,
        n_repeats=2,
        random_state=SEED,
        n_jobs=1,
    )
    importance_by_fold.append(result.importances_mean)

importance_summary = pd.DataFrame(
    {
        "feature": FEATURES,
        "mean P@50 drop when shuffled": np.mean(importance_by_fold, axis=0),
        "SD across folds": np.std(importance_by_fold, axis=0, ddof=1),
    }
).sort_values("mean P@50 drop when shuffled", ascending=False)
print(f"Permutation importance for {best_model_name}; positive means useful for ranking:")
print(importance_summary.to_string(index=False, float_format="%.4f"))

model_scores = oof_scores[best_model_name]
predicted_labels = (model_scores >= 0.5).astype(int)
actual_labels = y.to_numpy()
wrong = predicted_labels != actual_labels
false_positive = (predicted_labels == 1) & (actual_labels == 0)
false_negative = (predicted_labels == 0) & (actual_labels == 1)
print(
    f"\nAt the illustrative 0.5 cutoff: {false_positive.sum():,} false positives, "
    f"{false_negative.sum():,} false negatives across {len(df_data):,} out-of-fold rows."
)

error_indexes = np.flatnonzero(wrong)
error_margin = np.abs(model_scores[error_indexes] - 0.5)
example_indexes = error_indexes[np.argsort(-error_margin, kind="mergesort")[:3]]
error_rows = []
for index in example_indexes:
    is_false_positive = predicted_labels[index] == 1
    error_rows.append(
        {
            "error": "false positive" if is_false_positive else "false negative",
            "OOF score (not calibrated)": model_scores[index],
            "actual decline label": actual_labels[index],
            "March impressions": df_data.iloc[index]["gsc_impressions_march"],
            "March clicks": df_data.iloc[index]["gsc_clicks_march"],
            "March position": df_data.iloc[index]["gsc_avg_position_march"],
            "content age days": df_data.iloc[index]["content_age_days"],
            "GA4 available": df_data.iloc[index]["ga4_available_march"],
            "why this is hard": (
                "March signals ranked it high, but April did not cross the decline threshold."
                if is_false_positive
                else "March signals looked mild, but April crossed the decline threshold."
            ),
        }
    )
error_examples = pd.DataFrame(error_rows)
print("\nThree anonymous, most decisive out-of-fold mistakes:")
print(error_examples.to_string(index=False, float_format="%.3f"))

error_frame = pd.DataFrame(
    {
        "content_age_days": df_data["content_age_days"],
        "false_positive": false_positive,
        "false_negative": false_negative,
        "any_error": wrong,
    }
)
error_frame["age_band"] = pd.cut(
    error_frame["content_age_days"],
    bins=[-np.inf, 180, 365, np.inf],
    labels=["under 180 days", "180-365 days", "over 365 days"],
)
error_by_age = error_frame.groupby("age_band", observed=False).agg(
    rows=("any_error", "size"),
    error_rate=("any_error", "mean"),
    false_positives=("false_positive", "sum"),
    false_negatives=("false_negative", "sum"),
).reset_index()
print("\nOut-of-fold error mix by age (0.5 diagnostic cutoff):")
print(error_by_age.to_string(index=False, float_format="%.3f"))

feature_reading = {
    "gsc_impressions_march": "search visibility / opportunity size",
    "gsc_clicks_march": "recent search visits",
    "gsc_avg_position_march": "observed search ranking position",
    "ga4_sessions_march": "analytics sessions when GA4 was available",
    "content_age_days": "content age at the March decision date",
    "ga4_available_march": "whether GA4 measurement was available",
    "position_available_march": "whether a nonzero search position was observed",
}
print("\nInterpretation:")
for row in importance_summary.head(3).itertuples(index=False):
    print(
        f"- {row.feature} ({feature_reading[row.feature]}): "
        f"shuffling changed P@50 by {row[1]:+.4f} on average across held-out folds."
    )
p50_difference = model_p50[best_model_name] - rule_p50
if p50_difference > 0:
    print(
        f"On these folds, {best_model_name} was {p50_difference:.3f} higher than the "
        "Week-4 rule at mean P@50; this is directional evidence, not proof of future lift."
    )
else:
    print(
        f"On these folds, the Week-4 rule was {-p50_difference:.3f} higher than "
        f"{best_model_name} at mean P@50; added model complexity did not earn its place here."
    )
print(
    "False positives are pages that looked risky in March but did not meet the April "
    "20% impression-drop proxy; false negatives are declines without a strong March "
    "warning. A single next-month threshold is noisy, and none of these results show "
    "that refreshing a page changes its traffic."
)

Permutation importance for Logistic Regression; positive means useful for ranking:
                 feature  mean P@50 drop when shuffled  SD across folds
        gsc_clicks_march                        0.2840           0.1665
   gsc_impressions_march                        0.1400           0.1111
      ga4_sessions_march                        0.1200           0.1147
        content_age_days                        0.0780           0.1139
     ga4_available_march                        0.0200           0.0387
  gsc_avg_position_march                        0.0160           0.0635
position_available_march                        0.0000           0.0000

At the illustrative 0.5 cutoff: 37,681 false positives, 35,145 false negatives across 143,206 out-of-fold rows.

Three anonymous, most decisive out-of-fold mistakes:
         error  OOF score (not calibrated)  actual decline label  March impressions  March clicks  March position  content age days  GA4 available                            

## Self-check

Before submitting, confirm each line honestly:

- [x] The method, split, same-fold baseline comparison, and error interpretation are filled in.
- [ ] The notebook has run top to bottom in an authenticated Colab runtime with warehouse access.
- [x] The code excludes identifiers and April outcomes from model features; no synthetic-label fallback is used.
- [x] Claims are framed as observed, directional decision-support, not causal impact.
- [ ] I read the research paper linked in the assignment card and noted its methodology.
- [ ] The executed notebook is committed under `work/notebooks/`; then submit the repository URL on the card.

The local editor in this session has no Python runtime. In Colab, sign in, provide the gated dataset's `HF_TOKEN` through Secrets, then run all cells before submission.